In [0]:
storage_account_name = "f1datalakestorageme"
secret_scope = "storage-secrets"
secret_key = "f1-storage-account-key"
storage_account_access_key = dbutils.secrets.get(scope=secret_scope, key=secret_key)
container = "raw"
mount_point = "/mnt/f1-raw"

if not any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
    dbutils.fs.mount(
        source=f"wasbs://{container}@{storage_account_name}.blob.core.windows.net",
        mount_point=mount_point,
        extra_configs={
            f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": storage_account_access_key
        }
    )
    print("Mount successful")
else:
    print("This Storage Mount already exists")

In [0]:
files = dbutils.fs.ls(mount_point)
print(f"Found {len(files)} items in {mount_point}")

file_rows = [(f.path, f.name, f.size, f.modificationTime) for f in files]
file_df = spark.createDataFrame(file_rows, ["path", "name", "size", "modificationTime"])
display(file_df.limit(20))